In [8]:
import os
import subprocess
from pathlib import Path
import pandas as pd
from typing import List, Optional

# =========================
# CONFIG (EDIT THESE PATHS)
# =========================
CLONE_ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Clone")
BOUNDARY_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\List_Boundary_Commit_Events.csv")  # <-- change to where your csv is
OUTDIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5")
OUTDIR.mkdir(parents=True, exist_ok=True)

WORKFLOW_PATHS = [".github/workflows", ".github/main.workflow"]

# Git executable autodetect
GIT_EXE_CANDIDATES = [
    "git",
    r"C:\Program Files\Git\bin\git.exe",
    r"C:\Program Files\Git\cmd\git.exe",
    r"C:\Program Files (x86)\Git\bin\git.exe",
    r"C:\Program Files (x86)\Git\cmd\git.exe",
]

def pick_git_exe() -> str:
    for exe in GIT_EXE_CANDIDATES:
        try:
            r = subprocess.run([exe, "--version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            if r.returncode == 0 and "git version" in (r.stdout.lower() + r.stderr.lower()):
                return exe
        except FileNotFoundError:
            continue
    raise FileNotFoundError("Could not find git.exe. Install Git for Windows or add it to PATH.")

GIT_EXE = pick_git_exe()
print("Using git:", GIT_EXE)

def run_git(repo_dir: Path, args: List[str]) -> subprocess.CompletedProcess:
    return subprocess.run(
        [GIT_EXE, "-C", str(repo_dir), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

def is_git_repo(repo_dir: Path) -> bool:
    r = run_git(repo_dir, ["rev-parse", "--is-inside-work-tree"])
    return r.returncode == 0 and r.stdout.strip() == "true"

def commit_exists(repo_dir: Path, sha: str) -> bool:
    r = run_git(repo_dir, ["cat-file", "-e", f"{sha}^{{commit}}"])
    return r.returncode == 0

def actions_present_at_commit(repo_dir: Path, sha: str) -> Optional[bool]:
    """
    True  => at commit sha, at least one Actions workflow path exists
    False => commit exists, but no workflow paths exist at that commit
    None  => repo/commit missing locally or other git error
    """
    if not is_git_repo(repo_dir):
        return None
    if not commit_exists(repo_dir, sha):
        return None

    r = run_git(repo_dir, ["ls-tree", "-r", "--name-only", sha, "--", *WORKFLOW_PATHS])
    if r.returncode != 0:
        return None
    return bool(r.stdout.strip())

def first_actions_commit(repo_dir: Path) -> Optional[str]:
    r = run_git(repo_dir, ["rev-list", "--all", "--reverse", "-n", "1", "--", *WORKFLOW_PATHS])
    sha = r.stdout.strip() if r.returncode == 0 else ""
    return sha or None

def commit_iso_date(repo_dir: Path, sha: str) -> Optional[str]:
    r = run_git(repo_dir, ["show", "-s", "--format=%cI", sha])
    if r.returncode != 0:
        return None
    d = r.stdout.strip()
    return d if d else None

# =========================
# LOAD + PICK FIRST START BOUNDARY PER REPO
# =========================
df = pd.read_csv(BOUNDARY_CSV)

# Ensure types / sort keys
df["is_episode_start_boundary"] = df["is_episode_start_boundary"].astype(int)
df["episode_index"] = df["episode_index"].astype(int)
df["event_date_utc"] = pd.to_datetime(df["event_date_utc"], errors="coerce")

starts = df[df["is_episode_start_boundary"] == 1].copy()

# "first instrumentation episode" = min episode_index, tie-break by earliest date
starts = starts.sort_values(["repo_name", "episode_index", "event_date_utc"])
first_start = starts.groupby("repo_name", as_index=False).first()

print("Repos with at least one start boundary:", first_start["repo_name"].nunique())

# =========================
# CHECK ACTIONS PRESENCE AT THAT START COMMIT
# =========================
records = []
missing_repo_dirs = 0

for _, row in first_start.iterrows():
    repo_name = row["repo_name"]
    repo_dir = CLONE_ROOT / repo_name
    sha = str(row["commit_sha"]).strip()

    rec = {
        "repo_name": repo_name,
        "repo_dir": str(repo_dir),
        "full_name": row.get("full_name", ""),
        "first_episode_index": int(row["episode_index"]),
        "first_start_env_style": row.get("env_style", ""),
        "first_start_event_type": row.get("event_type", ""),
        "first_start_event_date_utc": str(row["event_date_utc"]) if pd.notna(row["event_date_utc"]) else "",
        "first_start_commit_sha": sha,
        "repo_found_locally": repo_dir.exists() and repo_dir.is_dir(),
        "is_git_repo": False,
        "actions_present_at_first_start_boundary": "",
        "first_actions_commit_in_history": "",
        "first_actions_date_in_history": "",
        "error": "",
    }

    if not rec["repo_found_locally"]:
        missing_repo_dirs += 1
        rec["error"] = "REPO_DIR_NOT_FOUND"
        records.append(rec)
        continue

    rec["is_git_repo"] = is_git_repo(repo_dir)
    if not rec["is_git_repo"]:
        rec["error"] = "NOT_A_GIT_REPO"
        records.append(rec)
        continue

    present = actions_present_at_commit(repo_dir, sha)
    if present is None:
        rec["actions_present_at_first_start_boundary"] = ""
        rec["error"] = "START_SHA_NOT_FOUND_LOCALLY_OR_GIT_ERROR"
    else:
        rec["actions_present_at_first_start_boundary"] = bool(present)

    fac = first_actions_commit(repo_dir)
    rec["first_actions_commit_in_history"] = fac or ""
    rec["first_actions_date_in_history"] = commit_iso_date(repo_dir, fac) if fac else ""

    records.append(rec)

out = pd.DataFrame(records)

# =========================
# WRITE OUTPUTS
# =========================
all_path = OUTDIR / "repos_first_episode_start_has_actions.csv"
out.to_csv(all_path, index=False)

true_subset = out[out["actions_present_at_first_start_boundary"] == True].copy()
true_path = OUTDIR / "repos_first_episode_start_has_actions_TRUE.csv"
true_subset.to_csv(true_path, index=False)

print("\nDone.")
print("Missing repo dirs:", missing_repo_dirs)
print("Total repos checked:", len(out))
print("Actions present at FIRST start boundary:", (out["actions_present_at_first_start_boundary"] == True).sum())
print("Wrote:\n -", all_path, "\n -", true_path)


Using git: git
Repos with at least one start boundary: 399

Done.
Missing repo dirs: 1
Total repos checked: 399
Actions present at FIRST start boundary: 201
Wrote:
 - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\repos_first_episode_start_has_actions.csv 
 - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\repos_first_episode_start_has_actions_TRUE.csv


In [11]:
# token check

In [13]:
from pathlib import Path
import requests
import re

ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

def load_env_tokens(env_path: Path):
    """
    Reads KEY=VALUE lines. Supports quoted values and ignores comments/blank lines.
    Returns dict of env vars found in file.
    """
    if not env_path.exists():
        raise FileNotFoundError(f"Env file not found: {env_path}")

    env = {}
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        env[k] = v
    return env

env = load_env_tokens(ENV_PATH)

# Collect your four tokens
token_keys = [f"GITHUB_TOKEN_{i}" for i in range(1, 5)]
tokens = [(k, env.get(k, "").strip()) for k in token_keys]

print("Loaded token lengths:")
for k, t in tokens:
    print(k, len(t))

def test_token(token: str):
    """
    Tests token validity. Returns (ok: bool, status_code: int, login: str|None).
    """
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "Authorization": f"token {token}",  # <— use 'token' form (works well with PATs)
    }
    r = requests.get("https://api.github.com/user", headers=headers, timeout=30)
    if r.status_code == 200:
        return True, r.status_code, r.json().get("login")
    return False, r.status_code, None

working = []
for k, t in tokens:
    if not t:
        print(f"{k}: EMPTY")
        continue
    ok, status, login = test_token(t)
    print(f"{k}: status={status}, ok={ok}, login={login}")
    if ok:
        working.append((k, t, login))

if not working:
    raise RuntimeError("No working tokens found. 401 means invalid/expired token OR wrong host (Enterprise).")

# Pick the first working token
ACTIVE_TOKEN_KEY, ACTIVE_TOKEN, ACTIVE_LOGIN = working[0]
print("\nUsing:", ACTIVE_TOKEN_KEY, "as", ACTIVE_LOGIN)


Loaded token lengths:
GITHUB_TOKEN_1 40
GITHUB_TOKEN_2 40
GITHUB_TOKEN_3 40
GITHUB_TOKEN_4 40
GITHUB_TOKEN_1: status=200, ok=True, login=BehnamParsa4
GITHUB_TOKEN_2: status=200, ok=True, login=BehnamParsa2
GITHUB_TOKEN_3: status=200, ok=True, login=BehnamAcc6
GITHUB_TOKEN_4: status=200, ok=True, login=BehnamParsa5

Using: GITHUB_TOKEN_1 as BehnamParsa4


In [4]:
# Step 1 by working token


In [16]:
import re
import time
import itertools
from pathlib import Path
import pandas as pd
import requests

# =========================
# PATHS / CONFIG
# =========================
INPUT_TRUE = Path(r"D:\5_RQ5\Pass1\repos_first_episode_start_has_actions_TRUE.csv")
OUTDIR     = Path(r"D:\5_RQ5\Pass1")
ENV_PATH   = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

OUTDIR.mkdir(parents=True, exist_ok=True)
OUT_ALL = OUTDIR / "gha_run_availability_screen.csv"
OUT_OK  = OUTDIR / "gha_metrics_eligible_repos.csv"

# =========================
# LOAD TOKENS (1..4)
# =========================
def load_env_tokens(env_path: Path):
    env = {}
    if not env_path.exists():
        raise FileNotFoundError(f"Env file not found: {env_path}")
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env_tokens(ENV_PATH)
token_keys = [f"GITHUB_TOKEN_{i}" for i in range(1, 5)]
tokens = [env.get(k, "").strip() for k in token_keys if env.get(k, "").strip()]
if not tokens:
    raise RuntimeError("No tokens found in All_Tokens.env (GITHUB_TOKEN_1..4).")

def token_ok(tok: str) -> bool:
    h = {"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28","Authorization":f"token {tok}"}
    r = requests.get("https://api.github.com/user", headers=h, timeout=30)
    return r.status_code == 200

working_tokens = [t for t in tokens if token_ok(t)]
print("Working tokens:", len(working_tokens))
if not working_tokens:
    raise RuntimeError("No working tokens in this session (all tokens failed /user).")

class TokenPool:
    def __init__(self, tokens):
        self.tokens = list(tokens)
        self.blocked_until = {t: 0 for t in self.tokens}
        self.cycle = itertools.cycle(self.tokens)

    def next_token(self):
        now = time.time()
        for _ in range(len(self.tokens)):
            t = next(self.cycle)
            if now >= self.blocked_until.get(t, 0):
                return t
        soonest = min(self.blocked_until.values())
        sleep_s = max(10, int(soonest - now) + 10)
        print(f"[all tokens rate-limited] sleeping {sleep_s}s...")
        time.sleep(sleep_s)
        return self.next_token()

    def block(self, token, reset_unix):
        if reset_unix and str(reset_unix).isdigit():
            self.blocked_until[token] = int(reset_unix)

POOL = TokenPool(working_tokens)

def gh_get(url, params=None, max_retries=6):
    last = None
    for attempt in range(max_retries):
        tok = POOL.next_token()
        headers = {
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Authorization": f"token {tok}",
        }
        r = requests.get(url, headers=headers, params=params, timeout=60)
        last = r

        if r.status_code == 200:
            return r

        rem = r.headers.get("X-RateLimit-Remaining")
        reset = r.headers.get("X-RateLimit-Reset")

        msg = ""
        try:
            if "application/json" in (r.headers.get("content-type") or ""):
                msg = (r.json().get("message") or "")
        except Exception:
            pass

        # Rate-limit: block only this token
        if r.status_code == 403 and (rem == "0" or "rate limit" in msg.lower()):
            POOL.block(tok, reset)
            continue

        # transient server errors
        if r.status_code in (500, 502, 503, 504):
            time.sleep(2 + attempt * 2)
            continue

        return r
    return last

# =========================
# HELPERS
# =========================
def normalize_slug(s: str) -> str:
    s = str(s or "").strip()
    s = s.replace("https://github.com/", "").replace("http://github.com/", "")
    return s.strip("/")

def repo_slug(row) -> str:
    # Prefer full_name if present
    fn = normalize_slug(row.get("full_name", ""))
    if fn and "/" in fn and " " not in fn:
        return fn

    # If owner_repo already exists in file
    orp = normalize_slug(row.get("owner_repo", ""))
    if orp and "/" in orp:
        return orp

    # Fallback: repo_name like Owner__Repo
    rn = str(row.get("repo_name", "")).strip()
    m = re.match(r"^([^_]+)__([^_]+)$", rn)
    if m:
        return f"{m.group(1)}/{m.group(2)}"

    return ""

# =========================
# LOAD INPUT
# =========================
df = pd.read_csv(INPUT_TRUE, dtype=str)

if "first_start_event_date_utc" not in df.columns:
    raise ValueError("Missing column 'first_start_event_date_utc' in repos_first_episode_start_has_actions_TRUE.csv")

# quick sanity check
t = gh_get("https://api.github.com/user")
print("Token sanity check:", t.status_code, (t.json().get("login") if t and t.status_code == 200 else "FAILED"))

records = []
n = len(df)

for i, row in df.iterrows():
    slug = repo_slug(row)
    start_dt_raw = str(row.get("first_start_event_date_utc", "")).strip()
    start_date = start_dt_raw.split(" ")[0].split("T")[0]  # YYYY-MM-DD

    rec = dict(row)
    rec["owner_repo"] = slug
    rec["start_date_yyyy_mm_dd"] = start_date
    rec["api_status"] = ""
    rec["api_note"] = ""
    rec["runs_total_since_start"] = ""

    if not slug or not start_date or start_date.lower() == "nan":
        rec["api_status"] = "SKIP_MISSING_SLUG_OR_DATE"
        records.append(rec)
        continue

    url = f"https://api.github.com/repos/{slug}/actions/runs"
    params = {"per_page": 1, "created": f">={start_date}"}

    r = gh_get(url, params=params)
    rec["api_status"] = (r.status_code if r is not None else "NO_RESPONSE")
    rec["rate_remaining"] = (r.headers.get("X-RateLimit-Remaining", "") if r is not None else "")
    rec["rate_reset"] = (r.headers.get("X-RateLimit-Reset", "") if r is not None else "")

    if r is not None and r.status_code == 200:
        j = r.json()
        rec["runs_total_since_start"] = j.get("total_count", 0)
    else:
        try:
            if r is not None and "application/json" in (r.headers.get("content-type") or ""):
                rec["api_note"] = (r.json().get("message", "")[:300])
            elif r is not None:
                rec["api_note"] = (r.text[:300])
            else:
                rec["api_note"] = "No response"
        except Exception:
            rec["api_note"] = "Could not parse error body"

    records.append(rec)

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{n} ...")
        time.sleep(0.7)

out = pd.DataFrame(records)

out["runs_total_since_start_num"] = pd.to_numeric(out["runs_total_since_start"], errors="coerce")

out.to_csv(OUT_ALL, index=False, encoding="utf-8")
eligible = out[(out["api_status"] == 200) & (out["runs_total_since_start_num"] > 0)].copy()
eligible.to_csv(OUT_OK, index=False, encoding="utf-8")

print("\nWrote:", OUT_ALL)
print("Wrote:", OUT_OK)
print("Eligible repos:", len(eligible), "out of", len(out))
print("\nStatus breakdown:")
print(out["api_status"].value_counts().head(20))


Working tokens: 3
Token sanity check: 200 BehnamParsa4
Processed 50/201 ...
Processed 100/201 ...
Processed 150/201 ...

Wrote: D:\5_RQ5\Pass1\gha_run_availability_screen.csv
Wrote: D:\5_RQ5\Pass1\gha_metrics_eligible_repos.csv
Eligible repos: 145 out of 201

Status breakdown:
api_status
200                          191
SKIP_MISSING_SLUG_OR_DATE     10
Name: count, dtype: int64


In [ ]:
#Step 2A: Collect runs_index.csv (repo → runs between start date and cutoff)

In [28]:
import time
import itertools
from pathlib import Path
import pandas as pd
import requests

# =========================
# PATHS / CONFIG
# =========================
BASE = Path(r"D:\5_RQ5\Pass1")
BASE.mkdir(parents=True, exist_ok=True)

ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

# Use sample OR full
ELIGIBLE = Path(r"D:\5_RQ5\Pass1\gha_metrics_eligible_repos_Sample.csv")  # change to gha_metrics_eligible_repos.csv when ready
OUTPUT   = BASE / "runs_index.csv"

PER_PAGE = 100
MAX_PAGES = 10   # GitHub search-style filters often effectively cap at 1000 results; keep explicit

# =========================
# TOKEN ROTATION (robust)
# =========================
def load_env_tokens(env_path: Path):
    env = {}
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env_tokens(ENV_PATH)
token_keys = [f"GITHUB_TOKEN_{i}" for i in range(1, 5)]
tokens = [env.get(k, "").strip() for k in token_keys if env.get(k, "").strip()]
if not tokens:
    raise RuntimeError("No tokens found in All_Tokens.env (GITHUB_TOKEN_1..4).")

def token_ok(tok: str) -> bool:
    h = {"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28","Authorization":f"token {tok}"}
    r = requests.get("https://api.github.com/user", headers=h, timeout=30)
    return r.status_code == 200

working_tokens = [t for t in tokens if token_ok(t)]
print("Working tokens:", len(working_tokens))
if not working_tokens:
    raise RuntimeError("No working tokens in this session (/user failed).")

class TokenPool:
    def __init__(self, tokens):
        self.tokens = list(tokens)
        self.blocked_until = {t: 0 for t in self.tokens}
        self.cycle = itertools.cycle(self.tokens)

    def next_token(self):
        now = time.time()
        for _ in range(len(self.tokens)):
            t = next(self.cycle)
            if now >= self.blocked_until.get(t, 0):
                return t
        soonest = min(self.blocked_until.values())
        sleep_s = max(10, int(soonest - now) + 10)
        print(f"[all tokens blocked] sleeping {sleep_s}s...")
        time.sleep(sleep_s)
        return self.next_token()

    def block_for(self, token, seconds: int):
        self.blocked_until[token] = int(time.time()) + int(seconds)

    def block_until_reset(self, token, reset_unix):
        if reset_unix and str(reset_unix).isdigit():
            self.blocked_until[token] = int(reset_unix)

POOL = TokenPool(working_tokens)
SESSION = requests.Session()

def gh_get(url, params=None, max_retries=6):
    last = None
    for attempt in range(max_retries):
        tok = POOL.next_token()
        headers = {
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Authorization": f"token {tok}",
        }
        r = SESSION.get(url, headers=headers, params=params, timeout=60)
        last = r

        if r.status_code == 200:
            return r

        rem = r.headers.get("X-RateLimit-Remaining")
        reset = r.headers.get("X-RateLimit-Reset")

        msg = ""
        try:
            if "application/json" in (r.headers.get("content-type") or ""):
                msg = (r.json().get("message") or "")
        except Exception:
            pass

        # Primary rate limit
        if r.status_code == 403 and (rem == "0" or "rate limit" in msg.lower()):
            POOL.block_until_reset(tok, reset)
            continue

        # Secondary/abuse limits: short block (rotate to next token)
        if r.status_code == 403 and ("secondary rate limit" in msg.lower() or "abuse" in msg.lower()):
            POOL.block_for(tok, 90)
            continue

        if r.status_code in (500, 502, 503, 504):
            time.sleep(2 + attempt * 2)
            continue

        return r
    return last

# =========================
# LOAD ELIGIBLE REPOS
# =========================
df = pd.read_csv(ELIGIBLE, dtype=str)
need_cols = {"owner_repo", "start_date_yyyy_mm_dd"}
missing = need_cols - set(df.columns)
if missing:
    raise ValueError(f"Eligible CSV missing columns: {missing}. Found: {list(df.columns)}")

print("Repos to process:", len(df))

# Resume: skip already indexed run_ids
seen_run_ids = set()
if OUTPUT.exists():
    try:
        prev = pd.read_csv(OUTPUT, usecols=["run_id"], dtype={"run_id": str})
        seen_run_ids = set(prev["run_id"].astype(str))
        print("Resume: already have run rows:", len(prev))
    except Exception:
        pass

buffer = []
def flush():
    global buffer
    if not buffer:
        return
    header = (not OUTPUT.exists()) or (OUTPUT.stat().st_size == 0)
    pd.DataFrame(buffer).to_csv(OUTPUT, mode="a", index=False, header=header, encoding="utf-8")
    buffer = []

# =========================
# FETCH RUNS (paged)
# =========================
for idx, row in df.iterrows():
    owner_repo = str(row["owner_repo"]).strip()
    start_date = str(row["start_date_yyyy_mm_dd"]).strip()

    if not owner_repo or not start_date:
        continue

    url = f"https://api.github.com/repos/{owner_repo}/actions/runs"
    page = 1
    total_added = 0
    truncated = False

    while True:
        params = {
            "per_page": PER_PAGE,
            "page": page,
            "created": f">={start_date}",
        }
        r = gh_get(url, params=params)
        if r is None or r.status_code != 200:
            break

        data = r.json()
        runs = data.get("workflow_runs", []) or []
        if not runs:
            break

        for run in runs:
            run_id = str(run.get("id", "")).strip()
            if not run_id or run_id in seen_run_ids:
                continue
            seen_run_ids.add(run_id)
            buffer.append({
                "owner_repo": owner_repo,
                "start_date_yyyy_mm_dd": start_date,
                "run_id": run_id,
                "run_number": run.get("run_number"),
                "run_attempt": run.get("run_attempt"),
                "event": run.get("event"),
                "status": run.get("status"),
                "conclusion": run.get("conclusion"),
                "created_at": run.get("created_at"),
                "updated_at": run.get("updated_at"),
                "run_started_at": run.get("run_started_at"),
                "head_sha": run.get("head_sha"),
                "head_branch": run.get("head_branch"),
                "workflow_id": run.get("workflow_id"),
                "path": run.get("path"),
                "html_url": run.get("html_url"),
            })
            total_added += 1

        if len(buffer) >= 20000:
            flush()

        page += 1
        if page > MAX_PAGES:
            truncated = True
            break
        if len(runs) < PER_PAGE:
            break

    if (idx + 1) % 5 == 0:
        flush()
        print(f"Processed repos {idx+1}/{len(df)} | added_runs={total_added} | truncated={truncated}")

flush()
print("Done. Wrote:", OUTPUT)


Working tokens: 2
Repos to process: 3
Done. Wrote: D:\5_RQ5\Pass1\runs_index.csv


In [ ]:
#Step 2B: Collect jobs_steps.csv (run_id → jobs + steps timestamps)

In [30]:
import time
import itertools
from pathlib import Path
import pandas as pd
import requests

# =========================
# PATHS
# =========================
ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

INPUT  = Path(r"D:\5_RQ5\Pass1\runs_index.csv")
OUTDIR = Path(r"D:\5_RQ5\Pass2")
OUTDIR.mkdir(parents=True, exist_ok=True)

JOBS_OUT  = OUTDIR / "jobs_index.csv"
STEPS_OUT = OUTDIR / "jobs_steps.csv"
DIAG_OUT  = OUTDIR / "jobs_jobs_diagnostics.csv"

# =========================
# Token loading + rotation (real)
# =========================
def load_env_tokens(env_path: Path):
    env = {}
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env_tokens(ENV_PATH)
token_keys = [f"GITHUB_TOKEN_{i}" for i in range(1, 5)]
tokens = [env.get(k, "").strip() for k in token_keys if env.get(k, "").strip()]

def token_ok(tok: str) -> bool:
    h = {"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28","Authorization":f"token {tok}"}
    r = requests.get("https://api.github.com/user", headers=h, timeout=30)
    return r.status_code == 200

working_tokens = [t for t in tokens if token_ok(t)]
print("Working tokens:", len(working_tokens))
if not working_tokens:
    raise RuntimeError("No working tokens for Pass 2B.")

class TokenPool:
    def __init__(self, tokens):
        self.tokens = list(tokens)
        self.blocked_until = {t: 0 for t in self.tokens}
        self.cycle = itertools.cycle(self.tokens)

    def next_token(self):
        now = time.time()
        for _ in range(len(self.tokens)):
            t = next(self.cycle)
            if now >= self.blocked_until.get(t, 0):
                return t
        soonest = min(self.blocked_until.values())
        sleep_s = max(10, int(soonest - now) + 10)
        print(f"[all tokens blocked] sleeping {sleep_s}s...")
        time.sleep(sleep_s)
        return self.next_token()

    def block_until_reset(self, token, reset_unix):
        if reset_unix and str(reset_unix).isdigit():
            self.blocked_until[token] = int(reset_unix)

    def block_for(self, token, seconds: int):
        self.blocked_until[token] = int(time.time()) + int(seconds)

POOL = TokenPool(working_tokens)
SESSION = requests.Session()

def gh_get(url, params=None, max_retries=6):
    last = None
    for attempt in range(max_retries):
        tok = POOL.next_token()
        headers = {
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Authorization": f"token {tok}",
        }
        r = SESSION.get(url, headers=headers, params=params, timeout=60)
        last = r

        if r.status_code == 200:
            return r

        rem = r.headers.get("X-RateLimit-Remaining")
        reset = r.headers.get("X-RateLimit-Reset")

        msg = ""
        try:
            if "application/json" in (r.headers.get("content-type") or ""):
                msg = (r.json().get("message") or "")
        except Exception:
            pass

        # Primary rate limit => block only that token
        if r.status_code == 403 and (rem == "0" or "rate limit" in msg.lower()):
            POOL.block_until_reset(tok, reset)
            continue

        # Secondary/abuse limit => short block only that token
        if r.status_code == 403 and ("secondary rate limit" in msg.lower() or "abuse" in msg.lower()):
            POOL.block_for(tok, 90)
            continue

        if r.status_code in (500, 502, 503, 504):
            time.sleep(2 + attempt * 2)
            continue

        return r
    return last

# =========================
# Load runs
# =========================
runs = pd.read_csv(INPUT, dtype={"run_id": str})
print("Runs loaded:", len(runs))

# Resume: skip runs already written to JOBS_OUT (job-level is your truth)
processed = set()
if JOBS_OUT.exists():
    try:
        prev = pd.read_csv(JOBS_OUT, usecols=["run_id"], dtype={"run_id": str})
        processed = set(prev["run_id"].astype(str).tolist())
        print(f"Resume: already processed {len(processed)} run_ids (jobs_index)")
    except Exception:
        pass

jobs_buf = []
steps_buf = []
diag_buf = []

def flush(buf, path: Path):
    if not buf:
        return
    header = (not path.exists()) or (path.stat().st_size == 0)
    pd.DataFrame(buf).to_csv(path, mode="a", index=False, header=header, encoding="utf-8")
    buf.clear()

added_jobs = 0
added_steps = 0
warn_failed = 0

# =========================
# Fetch jobs (+ steps if available)
# =========================
for i, rrow in runs.iterrows():
    owner_repo = str(rrow.get("owner_repo", "")).strip()
    run_id = str(rrow.get("run_id", "")).strip()
    if not owner_repo or not run_id or run_id in processed:
        continue

    url = f"https://api.github.com/repos/{owner_repo}/actions/runs/{run_id}/jobs"
    params = {"per_page": 100, "page": 1}

    jobs_count = 0
    steps_count = 0
    last_status = None
    last_note = ""

    while True:
        resp = gh_get(url, params=params)
        if resp is None:
            warn_failed += 1
            last_note = "NO_RESPONSE"
            break

        last_status = resp.status_code
        if resp.status_code != 200:
            warn_failed += 1
            try:
                last_note = resp.json().get("message", "")[:200]
            except Exception:
                last_note = (resp.text or "")[:200]
            break

        data = resp.json()
        jobs = data.get("jobs", []) or []
        if not jobs:
            break

        for job in jobs:
            jobs_count += 1
            job_id = job.get("id")
            job_name = job.get("name")
            job_started = job.get("started_at")
            job_completed = job.get("completed_at")
            job_conclusion = job.get("conclusion")
            job_status = job.get("status")
            runner_name = job.get("runner_name")
            runner_group = job.get("runner_group_name")

            job_labels = job.get("labels") or []
            if isinstance(job_labels, list):
                job_labels = ",".join([str(x) for x in job_labels])
            else:
                job_labels = str(job_labels)

            # ✅ ALWAYS write job-level
            jobs_buf.append({
                "owner_repo": owner_repo,
                "run_id": run_id,
                "job_id": job_id,
                "job_name": job_name,
                "job_status": job_status,
                "job_conclusion": job_conclusion,
                "job_started_at": job_started,
                "job_completed_at": job_completed,
                "runner_name": runner_name,
                "runner_group_name": runner_group,
                "job_labels": job_labels,
            })
            added_jobs += 1

            # step-level only if present
            steps = job.get("steps", None)
            if isinstance(steps, list) and steps:
                for step in steps:
                    steps_count += 1
                    steps_buf.append({
                        "owner_repo": owner_repo,
                        "run_id": run_id,
                        "job_id": job_id,
                        "job_name": job_name,
                        "step_name": step.get("name"),
                        "step_status": step.get("status"),
                        "step_conclusion": step.get("conclusion"),
                        "step_number": step.get("number"),
                        "step_started_at": step.get("started_at"),
                        "step_completed_at": step.get("completed_at"),
                    })
                    added_steps += 1

        if len(jobs) < 100:
            break
        params["page"] += 1

        if len(jobs_buf) >= 20000:
            flush(jobs_buf, JOBS_OUT)
        if len(steps_buf) >= 20000:
            flush(steps_buf, STEPS_OUT)

    diag_buf.append({
        "owner_repo": owner_repo,
        "run_id": run_id,
        "jobs_api_status": last_status,
        "jobs_api_note": last_note,
        "jobs_count": jobs_count,
        "steps_count": steps_count,
        "steps_present": bool(steps_count > 0),
    })

    processed.add(run_id)

    if (i + 1) % 200 == 0:
        flush(jobs_buf, JOBS_OUT)
        flush(steps_buf, STEPS_OUT)
        flush(diag_buf, DIAG_OUT)
        print(f"Processed {i+1}/{len(runs)} | added_jobs={added_jobs} | added_steps={added_steps} | warn_failed={warn_failed}")

flush(jobs_buf, JOBS_OUT)
flush(steps_buf, STEPS_OUT)
flush(diag_buf, DIAG_OUT)

print("Done.")
print("added_jobs:", added_jobs, "| added_steps:", added_steps, "| warn_failed:", warn_failed)
print("JOBS_OUT exists:", JOBS_OUT.exists(), "size:", (JOBS_OUT.stat().st_size if JOBS_OUT.exists() else None))
print("STEPS_OUT exists:", STEPS_OUT.exists(), "size:", (STEPS_OUT.stat().st_size if STEPS_OUT.exists() else None))
print("DIAG_OUT exists:", DIAG_OUT.exists(), "size:", (DIAG_OUT.stat().st_size if DIAG_OUT.exists() else None))


Working tokens: 2
Runs loaded: 2026
Processed 200/2026 | added_jobs=384 | added_steps=4644 | warn_failed=0
Processed 400/2026 | added_jobs=770 | added_steps=9339 | warn_failed=0
Processed 600/2026 | added_jobs=1168 | added_steps=13892 | warn_failed=0
Processed 800/2026 | added_jobs=1572 | added_steps=18588 | warn_failed=0
Processed 1000/2026 | added_jobs=1948 | added_steps=22995 | warn_failed=0
Processed 1200/2026 | added_jobs=2329 | added_steps=26361 | warn_failed=0
Processed 1400/2026 | added_jobs=2539 | added_steps=27385 | warn_failed=0
Processed 1600/2026 | added_jobs=2767 | added_steps=28773 | warn_failed=0
Processed 1800/2026 | added_jobs=3007 | added_steps=29569 | warn_failed=0
Processed 2000/2026 | added_jobs=3250 | added_steps=29569 | warn_failed=0
Done.
added_jobs: 3281 | added_steps: 29569 | warn_failed: 0
JOBS_OUT exists: True size: 552424
STEPS_OUT exists: True size: 4269214
DIAG_OUT exists: True size: 95324


In [ ]:
#Step 2C: Identify instrumentation steps + compute durations + attach style label

In [1]:
# =========================
# Pass 3: job log labeling (YAML-detector rules applied to log text) — ROBUST
# =========================
import re
import time
import io
import zipfile
import itertools
from pathlib import Path
import pandas as pd
import requests

# -------------------------
# PATHS
# -------------------------
ENV_PATH  = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
JOBS_INDEX = Path(r"D:\5_RQ5\Pass2\jobs_index.csv")   # from Pass 2B
OUTDIR     = Path(r"D:\5_RQ5\Pass3")
OUTDIR.mkdir(parents=True, exist_ok=True)
OUT_LABELS = OUTDIR / "job_labels_from_logs.csv"

# -------------------------
# NETWORK / DOWNLOAD KNOBS
# -------------------------
CONNECT_TIMEOUT = 30
READ_TIMEOUT    = 300   # logs can be large
MAX_LOG_ZIP_BYTES = 60 * 1024 * 1024   # 60MB safety cap (tune if needed)
CHUNK_SIZE = 1024 * 256               # 256KB

# -------------------------
# TOKEN ROTATION (robust)
# -------------------------
def load_env_tokens(env_path: Path):
    env = {}
    if not env_path.exists():
        raise FileNotFoundError(f"Env file not found: {env_path}")
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env_tokens(ENV_PATH)
token_keys = [f"GITHUB_TOKEN_{i}" for i in range(1, 5)]
tokens = [env.get(k, "").strip() for k in token_keys if env.get(k, "").strip()]
if not tokens:
    raise RuntimeError("No tokens found in All_Tokens.env (GITHUB_TOKEN_1..4).")

def token_ok(tok: str) -> bool:
    h = {"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28","Authorization":f"token {tok}",
         "User-Agent":"rq5-pass3"}
    try:
        r = requests.get("https://api.github.com/user", headers=h, timeout=(CONNECT_TIMEOUT, 30))
        return r.status_code == 200
    except Exception:
        return False

working_tokens = [t for t in tokens if token_ok(t)]
print("Working tokens:", len(working_tokens))
if not working_tokens:
    raise RuntimeError("No working tokens (/user failed).")

class TokenPool:
    def __init__(self, tokens):
        self.tokens = list(tokens)
        self.blocked_until = {t: 0 for t in self.tokens}
        self.cycle = itertools.cycle(self.tokens)

    def next_token(self):
        now = time.time()
        for _ in range(len(self.tokens)):
            t = next(self.cycle)
            if now >= self.blocked_until.get(t, 0):
                return t
        soonest = min(self.blocked_until.values())
        sleep_s = max(10, int(soonest - now) + 10)
        print(f"[all tokens blocked] sleeping {sleep_s}s...")
        time.sleep(sleep_s)
        return self.next_token()

    def block_for(self, token, seconds: int):
        self.blocked_until[token] = int(time.time()) + int(seconds)

    def block_until_reset(self, token, reset_unix):
        if reset_unix and str(reset_unix).isdigit():
            self.blocked_until[token] = int(reset_unix)

POOL = TokenPool(working_tokens)
SESSION = requests.Session()

def gh_get(url, params=None, max_retries=8, stream=False):
    """
    Robust GET with:
      - token rotation
      - retry on network errors (RemoteDisconnected / ConnectionError / timeout)
      - rate-limit blocking per token
    """
    last = None
    for attempt in range(max_retries):
        tok = POOL.next_token()
        headers = {
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Authorization": f"token {tok}",
            "User-Agent": "rq5-pass3",
        }
        try:
            r = SESSION.get(
                url,
                headers=headers,
                params=params,
                timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
                stream=stream,
                allow_redirects=True
            )
            last = r
        except requests.exceptions.RequestException as e:
            # THIS is where your RemoteDisconnected ends up
            sleep_s = min(60, 2 + attempt * 4)
            print(f"[net err] {type(e).__name__}: {e} | retrying in {sleep_s}s ...")
            time.sleep(sleep_s)
            continue

        if r.status_code == 200:
            return r

        rem = r.headers.get("X-RateLimit-Remaining")
        reset = r.headers.get("X-RateLimit-Reset")

        msg = ""
        try:
            if "application/json" in (r.headers.get("content-type") or ""):
                msg = (r.json().get("message") or "")
        except Exception:
            pass

        if r.status_code == 403 and (rem == "0" or "rate limit" in msg.lower()):
            POOL.block_until_reset(tok, reset)
            continue

        if r.status_code == 403 and ("secondary rate limit" in msg.lower() or "abuse" in msg.lower()):
            POOL.block_for(tok, 90)
            continue

        if r.status_code in (500, 502, 503, 504):
            time.sleep(2 + attempt * 2)
            continue

        # non-retryable HTTP response
        return r

    return last

# -------------------------
# DETECTOR (your trimmed rules)
# -------------------------
def compile_any(patterns, flags=re.I | re.M):
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns, text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq):
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def collect_hits_with_groups(patterns, text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl)
            groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

DEVICE_SOURCES = [
    ("Real_Device", "adb -s <serial> (physical)", [
        r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b'
    ]),
    ("Emulator", "adb -s emulator-serial", [r'(?mi)\badb\s+-s\s+(?:emulator-\d+)\b']),
    ("Emulator", "adb wait-for-device",   [r'(?mi)\badb\s+wait[- ]?for[- ]?device\b']),
    ("Emulator", "emulator -avd/@",       [r'(?mi)\bemulator\b[^\n]*-avd\s+\S+']),
    ("Emulator", "reactivecircus runner", [r'(?mi)reactivecircus/android-emulator-runner@']),
    ("Emulator", "malinskiy runner",      [r'(?mi)malinskiy/action-android/emulator-run-cmd@']),
    ("Third_Party_Lab", "gcloud firebase", [r'(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "appcenter test",  [r'(?mi)\bappcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "emulator.wtf action", [r'(?mi)\bemulator\.wtf\b']),
]
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

GRADLE_ANY = r'(?i)\bgradle(?:w)?(?:\.bat)?\b'
TRIGGER_SOURCES = [
    ("Gradle", "connectedAndroidTest", [rf'(?mi){GRADLE_ANY}[^\n]*\bconnectedandroidtest\b']),
    ("Gradle", "connectedCheck",       [rf'(?mi){GRADLE_ANY}[^\n]*\bconnectedcheck\b']),
    ("Gradle", "deviceCheck",          [rf'(?mi){GRADLE_ANY}[^\n]*\b(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest", [rf'(?mi){GRADLE_ANY}[^\n]*\b(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("ADB",    "am instrument",        [r'(?mi)\bam\s+instrument\b']),
    ("Third_Party_Lab", "gcloud firebase (instr)", [r'(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run[^\n]*(--test\b|--type\s+instrumentation\b)']),
    ("Third_Party_Lab", "flank",        [r'(?mi)\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",     [r'(?mi)\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "emulator.wtf run", [r'(?mi)\bemulator\.wtf\b']),
]
TRIGGER_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES]

FLUTTER_IT_LINE = re.compile(r'(?mi)\bflutter\s+(?:test|drive)\b[^\n]*')
FLUTTER_HINTS   = re.compile(r'(?i)(integration_test|--driver\b|/integration_test/)')

ANDROID_RUNTIME_ENV_LABELS = {
    "emulator -avd/@",
    "adb wait-for-device",
    "adb -s emulator-serial",
    "reactivecircus runner",
    "malinskiy runner",
    "adb -s <serial> (physical)",
}
ANDROID_STRICT_3P_LABELS = {"gcloud firebase", "appcenter test", "emulator.wtf action"}
_ANDROID_RUNTIME_ENV_LABELS_L = {s.lower() for s in ANDROID_RUNTIME_ENV_LABELS}
_ANDROID_STRICT_3P_LABELS_L   = {s.lower() for s in ANDROID_STRICT_3P_LABELS}

def has_android_runtime_evidence(dev_labels, dev_groups) -> bool:
    lbls = {str(l).strip().lower() for l in (dev_labels or []) if str(l).strip()}
    grps = {str(g).strip() for g in (dev_groups or []) if str(g).strip()}
    if "Real_Device" in grps:
        return True
    if lbls & _ANDROID_RUNTIME_ENV_LABELS_L:
        return True
    if lbls & _ANDROID_STRICT_3P_LABELS_L:
        return True
    return False

def has_gmd_gradle_trigger(trigger_labels):
    L = {l.lower() for l in (trigger_labels or [])}
    return any("manageddevice androidtest" in l for l in L)

def map_test_invocations(groups, trigger_labels, flutter_androidish):
    out = []
    if has_gmd_gradle_trigger(trigger_labels):
        out.append("Gradle_GMD")
    if any(l.lower() in {"connectedandroidtest","connectedcheck","devicecheck"} or "connected" in l.lower() for l in (trigger_labels or [])):
        out.append("Gradle_Connected")
    if "ADB" in (groups or []):
        out.append("ADB")
    if "Third_Party_Lab" in (groups or []):
        out.append("3P CLIs")
    if flutter_androidish and any(l.lower() == "flutter integration test" for l in (trigger_labels or [])):
        out.append("3P CLIs")
    return sorted(set(out))

def map_execution_envs(groups, labels):
    envs = set()
    s_groups, s_labels = set(groups or []), set(labels or [])
    if "Third_Party_Lab" in s_groups:
        envs.add("Third Party")
    if "Real_Device" in s_groups:
        envs.add("Real Device")
    if "Emulator" in s_groups:
        if "reactivecircus runner" in s_labels:
            envs.add("Emulator_ReactiveCircus")
        elif "malinskiy runner" in s_labels:
            envs.add("Emulator_Malinskiy")
        else:
            envs.add("Emulator_DIY")
    return sorted(envs)

def map_exec_env_style(exec_envs):
    styles = set()
    if any(e.startswith("Emulator_") for e in (exec_envs or [])):
        if "Emulator_DIY" in exec_envs:
            styles.add("Emu_Custom")
        if any(e in {"Emulator_ReactiveCircus","Emulator_Malinskiy"} for e in exec_envs):
            styles.add("Emu_Community")
    if "Third Party" in (exec_envs or []):
        styles.add("Third-Party")
    if "Real Device" in (exec_envs or []):
        styles.add("Real Device")
    if "GMD" in (exec_envs or []):
        styles.add("GMD")
    return ",".join(sorted(styles)) if styles else ""

def map_test_inv_style(test_inv):
    toks = {t.strip() for t in str(test_inv).split(",") if t.strip()}
    styles = set()
    if any(t.startswith("Gradle") for t in toks):
        styles.add("Gradle-based")
    if "3P CLIs" in toks:
        styles.add("Third-Party CLI")
    if "ADB" in toks:
        styles.add("ADB")
    return ",".join(sorted(styles)) if styles else ""

def scan_log_text(text: str):
    low = (text or "").lower()
    dev_labels, dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, low)
    trig_labels, trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS, low)

    flutter_it_present = False
    flutter_androidish = False
    flutter_hits = []
    for m in FLUTTER_IT_LINE.finditer(text or ""):
        line = m.group(0)
        if FLUTTER_HINTS.search(line):
            flutter_hits.append(line)
    if flutter_hits:
        flutter_it_present = True
        trig_labels = unique_preserve(trig_labels + ["flutter integration test"])
        trig_groups = unique_preserve(trig_groups + ["Flutter"])

    has_android_env_strict = has_android_runtime_evidence(dev_labels, dev_groups)
    if flutter_hits and has_android_env_strict:
        flutter_androidish = True

    combined_groups = unique_preserve(trig_groups + dev_groups)
    non_flutter_triggers = [l for l in trig_labels if l.lower() != "flutter integration test"]
    instru_signal = bool(non_flutter_triggers) or bool(dev_groups) or (flutter_it_present and flutter_androidish)

    test_inv_list = map_test_invocations(combined_groups, trig_labels, flutter_androidish)
    test_inv = ",".join(test_inv_list)

    exec_envs = map_execution_envs(dev_groups, dev_labels)
    if has_gmd_gradle_trigger(trig_labels) and "GMD" not in exec_envs:
        exec_envs = exec_envs + ["GMD"]

    exec_envs_str = ",".join(exec_envs)
    exec_style = map_exec_env_style(exec_envs)
    inv_style  = map_test_inv_style(test_inv)

    return {
        "instru_job_signal": bool(instru_signal),
        "trigger_labels": ",".join(trig_labels),
        "trigger_groups": ",".join(trig_groups),
        "device_labels": ",".join(dev_labels),
        "device_groups": ",".join(dev_groups),
        "flutter_integ_t_signal": bool(flutter_it_present),
        "flutter_androidish": bool(flutter_androidish),
        "execution_environment": exec_envs_str,
        "test_invocation": test_inv,
        "Exec_Env_Style": exec_style,
        "Test_Inv_Style": inv_style,
    }

# -------------------------
# Download & read job logs (ROBUST)
# -------------------------
def fetch_job_logs_text(owner_repo: str, job_id: str):
    url = f"https://api.github.com/repos/{owner_repo}/actions/jobs/{job_id}/logs"

    r = gh_get(url, stream=True)
    if r is None:
        return "", "NO_RESPONSE"
    if r.status_code != 200:
        return "", f"HTTP_{r.status_code}"

    # Stream ZIP bytes safely (avoid r.content for huge files)
    buf = io.BytesIO()
    total = 0
    try:
        for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
            if not chunk:
                continue
            total += len(chunk)
            if total > MAX_LOG_ZIP_BYTES:
                return "", "TOO_LARGE"
            buf.write(chunk)
    except requests.exceptions.RequestException as e:
        return "", f"STREAM_ERR_{type(e).__name__}"

    data = buf.getvalue()
    if not data:
        return "", "EMPTY_LOGS"

    # Parse ZIP to text
    try:
        z = zipfile.ZipFile(io.BytesIO(data))
        texts = []
        for nm in z.namelist():
            try:
                b = z.read(nm)
                if b:
                    texts.append(b.decode("utf-8", errors="ignore"))
            except Exception:
                pass
        return "\n\n".join(texts), "OK"
    except Exception:
        # sometimes it's not a zip; fallback
        try:
            return data.decode("utf-8", errors="ignore"), "OK_TEXT"
        except Exception:
            return "", "BAD_ZIP"

# -------------------------
# MAIN
# -------------------------
jobs = pd.read_csv(JOBS_INDEX, dtype=str)
need = {"owner_repo","run_id","job_id","job_name"}
missing = need - set(jobs.columns)
if missing:
    raise ValueError(f"jobs_index.csv missing columns: {missing}. Found={list(jobs.columns)}")

# Resume
done = set()
if OUT_LABELS.exists():
    try:
        prev = pd.read_csv(OUT_LABELS, usecols=["job_id"], dtype=str)
        done = set(prev["job_id"].astype(str))
        print("Resume: already labeled jobs:", len(done))
    except Exception:
        pass

rows = []
def flush():
    global rows
    if not rows:
        return
    header = (not OUT_LABELS.exists()) or (OUT_LABELS.stat().st_size == 0)
    pd.DataFrame(rows).to_csv(OUT_LABELS, mode="a", index=False, header=header, encoding="utf-8")
    rows = []

n = len(jobs)
labeled = 0
skipped = 0
warn = 0

for i, r in jobs.iterrows():
    owner_repo = str(r["owner_repo"]).strip()
    run_id     = str(r["run_id"]).strip()
    job_id     = str(r["job_id"]).strip()
    job_name   = str(r.get("job_name","")).strip()

    if not owner_repo or not job_id:
        skipped += 1
        continue
    if job_id in done:
        skipped += 1
        continue

    text, st = fetch_job_logs_text(owner_repo, job_id)
    if st != "OK" and st != "OK_TEXT":
        warn += 1

    det = scan_log_text(text if (st == "OK" or st == "OK_TEXT") else "")
    rows.append({
        "owner_repo": owner_repo,
        "run_id": run_id,
        "job_id": job_id,
        "job_name": job_name,
        "logs_status": st,
        **det
    })
    labeled += 1
    done.add(job_id)

    if labeled % 100 == 0:
        print(f"Labeled {labeled}/{n} jobs | warn={warn} | skipped={skipped}")
        flush()

flush()
print("Done. Wrote:", OUT_LABELS)
print("labeled:", labeled, "warn:", warn, "skipped:", skipped)
print("logs_status counts (sample):")
try:
    tmp = pd.read_csv(OUT_LABELS, usecols=["logs_status"], dtype=str)
    print(tmp["logs_status"].value_counts().head(15))
except Exception:
    pass


Working tokens: 2
Labeled 100/3281 jobs | warn=24 | skipped=0
[net err] ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')) | retrying in 2s ...
Labeled 200/3281 jobs | warn=47 | skipped=0


KeyboardInterrupt: 

In [ ]:
# Final Pass 2C

In [27]:
import pandas as pd
from pathlib import Path

# =========================
# CONFIG (EDIT IF NEEDED)
# =========================
BASE = Path(r"D:\5_RQ5")                 # root
PASS1 = BASE / "Pass1"
PASS2 = BASE / "Pass2"
PASS3 = BASE / "Pass3"                   # adjust if your Pass3 folder differs

RUNS_INDEX  = PASS1 / "runs_index.csv"   # from Pass 2A
JOBS_INDEX  = PASS2 / "jobs_index.csv"   # from hybrid Pass 2B
JOBS_STEPS  = PASS2 / "jobs_steps.csv"   # from hybrid Pass 2B

# Pass 3 output (adjust filename if yours differs)
# expected: has at least owner_repo + run_id + job_id + something like "instru_t_ci_signal" OR "is_instrumentation_job"
PASS3_LABELS = PASS3 / "job_labels_from_logs.csv"

OUTDIR = BASE / "Pass4"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_RUN     = OUTDIR / "perf_run_level.csv"
OUT_JOB     = OUTDIR / "perf_job_level.csv"
OUT_STEP    = OUTDIR / "perf_step_level.csv"
OUT_IJOB    = OUTDIR / "perf_instru_job_level.csv"
OUT_IRUN    = OUTDIR / "perf_instru_run_level.csv"
OUT_BEST    = OUTDIR / "perf_best_available.csv"

# =========================
# HELPERS
# =========================
def to_dt(s):
    return pd.to_datetime(s, errors="coerce", utc=True)

def coalesce_cols(df, candidates):
    """Return first existing column name from candidates, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def truthy_series(x):
    """Convert common truthy strings/bools to boolean Series."""
    if x is None:
        return None
    if x.dtype == bool:
        return x
    s = x.astype(str).str.strip().str.lower()
    return s.isin(["1","true","t","yes","y","ok"])

# =========================
# LOAD RUNS (Pass 2A)
# =========================
if not RUNS_INDEX.exists():
    raise FileNotFoundError(f"Missing RUNS_INDEX: {RUNS_INDEX}")

runs = pd.read_csv(RUNS_INDEX, dtype=str)

# timestamps (run-level)
runs["created_at_dt"]    = to_dt(runs.get("created_at"))
runs["updated_at_dt"]    = to_dt(runs.get("updated_at"))
runs["run_started_dt"]   = to_dt(runs.get("run_started_at"))

runs["run_start_dt"] = runs["run_started_dt"].fillna(runs["created_at_dt"])
runs["run_end_dt"]   = runs["updated_at_dt"]

runs["run_duration_s"] = (runs["run_end_dt"] - runs["run_start_dt"]).dt.total_seconds()

run_perf = runs[[
    "owner_repo","run_id","run_start_dt","run_end_dt","run_duration_s",
    "path","head_sha","head_branch"
]].copy()

run_perf.to_csv(OUT_RUN, index=False, encoding="utf-8")
print("Wrote:", OUT_RUN, "rows=", len(run_perf))

# =========================
# LOAD JOBS (Pass 2B hybrid)
# =========================
job_perf = None
if JOBS_INDEX.exists() and JOBS_INDEX.stat().st_size > 0:
    jobs = pd.read_csv(JOBS_INDEX, dtype=str)

    jobs["job_started_dt"]   = to_dt(jobs.get("job_started_at"))
    jobs["job_completed_dt"] = to_dt(jobs.get("job_completed_at"))
    jobs["job_duration_s"]   = (jobs["job_completed_dt"] - jobs["job_started_dt"]).dt.total_seconds()

    # keep common useful columns if present
    keep = [c for c in [
        "owner_repo","run_id","job_id","job_name",
        "job_started_dt","job_completed_dt","job_duration_s",
        "job_conclusion","job_status","job_labels","runner_name","runner_group_name"
    ] if c in jobs.columns]

    job_perf = jobs[keep].copy()
    job_perf.to_csv(OUT_JOB, index=False, encoding="utf-8")
    print("Wrote:", OUT_JOB, "rows=", len(job_perf))
else:
    print("No JOBS_INDEX found (job-level perf will be skipped).")

# =========================
# LOAD STEPS (Pass 2B hybrid)
# =========================
step_perf = None
if JOBS_STEPS.exists() and JOBS_STEPS.stat().st_size > 0:
    steps = pd.read_csv(JOBS_STEPS, dtype=str)

    steps["step_started_dt"]   = to_dt(steps.get("step_started_at"))
    steps["step_completed_dt"] = to_dt(steps.get("step_completed_at"))
    steps["step_duration_s"]   = (steps["step_completed_dt"] - steps["step_started_dt"]).dt.total_seconds()

    keep = [c for c in [
        "owner_repo","run_id","job_id","job_name","step_number","step_name",
        "step_started_dt","step_completed_dt","step_duration_s",
        "step_conclusion","step_status",
        "job_labels","runner_name","runner_group_name"
    ] if c in steps.columns]

    step_perf = steps[keep].copy()
    step_perf.to_csv(OUT_STEP, index=False, encoding="utf-8")
    print("Wrote:", OUT_STEP, "rows=", len(step_perf))
else:
    print("No JOBS_STEPS found (step-level perf will be skipped).")

# =========================
# LOAD PASS 3 LABELS (instrumentation jobs)
# =========================
labels = None
if PASS3_LABELS.exists() and PASS3_LABELS.stat().st_size > 0:
    labels = pd.read_csv(PASS3_LABELS, dtype=str)

    # pick a label column that indicates "instrumentation job"
    # support multiple names you may have used
    label_col = coalesce_cols(labels, [
        "is_instrumentation_job",
        "instru_job_signal",
        "instru_t_ci_signal",
        "instru_t_job_signal",
        "instrumentation_job",
    ])

    if label_col is None:
        print("Pass 3 labels file found, but no recognizable instrumentation flag column.")
        labels = None
    else:
        labels["_is_instr"] = truthy_series(labels[label_col])
        # keep keys and any helpful detector columns if present
        keep = [c for c in [
            "owner_repo","run_id","job_id","job_name",
            label_col,
            "test_invocation","execution_environment","Exec_Env_Style","Test_Inv_Style",
            "gmd_config_signal","called_instru_t_ci_signal"
        ] if c in labels.columns]
        labels = labels[keep + ["_is_instr"]].copy()
        print("Loaded Pass 3 labels:", len(labels), "rows | flag col =", label_col)
else:
    print("No Pass 3 labels file found (instrumentation rollups will be skipped).")

# =========================
# INSTRUMENTATION JOB PERF (job-level only)
# =========================
ijob_perf = None
if (labels is not None) and (job_perf is not None) and len(job_perf) > 0:
    # merge on owner_repo + run_id + job_id (strongest join)
    merged = job_perf.merge(
        labels[["owner_repo","run_id","job_id","_is_instr"]],
        on=["owner_repo","run_id","job_id"],
        how="left"
    )
    merged["_is_instr"] = merged["_is_instr"].fillna(False)

    ijob_perf = merged[merged["_is_instr"]].copy()
    ijob_perf.to_csv(OUT_IJOB, index=False, encoding="utf-8")
    print("Wrote:", OUT_IJOB, "rows=", len(ijob_perf))
else:
    print("Instrumentation job-level perf skipped (need Pass3 labels + jobs_index).")

# =========================
# INSTRUMENTATION RUN PERF (rollup job durations per run)
# =========================
irun_perf = None
if ijob_perf is not None and len(ijob_perf) > 0:
    # rollup: sum job duration (best proxy for instrumentation time)
    agg = (ijob_perf.dropna(subset=["job_duration_s"])
                    .groupby(["owner_repo","run_id"], as_index=False)
                    .agg(
                        instru_job_time_s=("job_duration_s","sum"),
                        instru_job_count=("job_id","nunique"),
                    ))
    # attach run timestamps for convenience
    irun_perf = run_perf.merge(agg, on=["owner_repo","run_id"], how="left")
    irun_perf.to_csv(OUT_IRUN, index=False, encoding="utf-8")
    print("Wrote:", OUT_IRUN, "rows=", len(irun_perf))
else:
    print("Instrumentation run-level perf skipped (no instru jobs).")

# =========================
# BEST AVAILABLE (Step > Job > Run)
# + instrumentation-aware if available
# =========================
best = run_perf.copy()

# generic best metric (not instrumentation-specific)
best["metric_level"] = "run"
best["metric_name"] = "run_duration_s"
best["metric_value_s"] = best["run_duration_s"]

if job_perf is not None and len(job_perf) > 0:
    job_agg = (job_perf.dropna(subset=["job_duration_s"])
                      .groupby(["owner_repo","run_id"], as_index=False)["job_duration_s"]
                      .max()
                      .rename(columns={"job_duration_s":"job_max_duration_s"}))
    best = best.merge(job_agg, on=["owner_repo","run_id"], how="left")
    m = best["job_max_duration_s"].notna()
    best.loc[m, "metric_level"] = "job"
    best.loc[m, "metric_name"] = "job_max_duration_s"
    best.loc[m, "metric_value_s"] = best.loc[m, "job_max_duration_s"]

if step_perf is not None and len(step_perf) > 0:
    step_agg = (step_perf.dropna(subset=["step_duration_s"])
                       .groupby(["owner_repo","run_id"], as_index=False)["step_duration_s"]
                       .max()
                       .rename(columns={"step_duration_s":"step_max_duration_s"}))
    best = best.merge(step_agg, on=["owner_repo","run_id"], how="left")
    m = best["step_max_duration_s"].notna()
    best.loc[m, "metric_level"] = "step"
    best.loc[m, "metric_name"] = "step_max_duration_s"
    best.loc[m, "metric_value_s"] = best.loc[m, "step_max_duration_s"]

# If instrumentation rollup exists, include it as extra columns + an "instru_best" suggestion
if irun_perf is not None and len(irun_perf) > 0:
    best = best.merge(
        irun_perf[["owner_repo","run_id","instru_job_time_s","instru_job_count"]],
        on=["owner_repo","run_id"],
        how="left"
    )
    # suggested instrumentation metric: job sum (until you label step names in Pass 3+)
    best["instru_metric_name"] = None
    best["instru_metric_value_s"] = None
    m = best["instru_job_time_s"].notna()
    best.loc[m, "instru_metric_name"] = "instru_job_time_s"
    best.loc[m, "instru_metric_value_s"] = best.loc[m, "instru_job_time_s"]

best.to_csv(OUT_BEST, index=False, encoding="utf-8")
print("Wrote:", OUT_BEST, "rows=", len(best))
print("Best metric_level counts:\n", best["metric_level"].value_counts(dropna=False))
if "instru_metric_name" in best.columns:
    print("Has instru_metric rows:", best["instru_metric_name"].notna().sum())


Wrote: D:\5_RQ5\Pass4\perf_run_level.csv rows= 1084
Wrote: D:\5_RQ5\Pass4\perf_job_level.csv rows= 2308
Wrote: D:\5_RQ5\Pass4\perf_step_level.csv rows= 10547
Loaded Pass 3 labels: 2308 rows | flag col = instru_job_signal
Wrote: D:\5_RQ5\Pass4\perf_instru_job_level.csv rows= 141
Wrote: D:\5_RQ5\Pass4\perf_instru_run_level.csv rows= 1084
Wrote: D:\5_RQ5\Pass4\perf_best_available.csv rows= 1084
Best metric_level counts:
 metric_level
job     661
step    421
run       2
Name: count, dtype: int64
Has instru_metric rows: 141
